In [12]:
############################################################
# SMART RESUME SCREENING SYSTEM
# BERT + RAG + FAISS
# Best Performance Pipeline
############################################################

############################################################
# 1. IMPORT LIBRARIES
############################################################

import pandas as pd
import numpy as np
import re
import nltk
import torch
import faiss
import spacy

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sentence_transformers import SentenceTransformer

nltk.download('stopwords')

from nltk.corpus import stopwords



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\POOJA\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [13]:
###########################################################
# 2. LOAD DATASET
############################################################

train = pd.read_csv("Dataset/train.csv")
test = pd.read_csv("Dataset/test.csv")

print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

print(train.head())


Train Shape: (1986, 2)
Test Shape: (497, 2)
                                                text          label
0  ENGINEERING CONSULTANT Professional Summary To...     CONSULTANT
1  CARPENTER Summary Carpenter Foreman Position w...   CONSTRUCTION
2  DIGITAL MARKETING SPECIALIST Summary Digital m...  DIGITAL-MEDIA
3  SOUS CHEF Work Experience Sous Chef Jul 2010 C...           CHEF
4  DONOR ADVOCATE Professional Summary Organized ...       ADVOCATE


In [14]:

############################################################
# 3. BEST TEXT PREPROCESSING
############################################################

stop_words = set(stopwords.words("english"))

nlp = spacy.load("en_core_web_sm")


def clean_text(text):

    text = text.lower()

    text = re.sub(r'\S+@\S+', ' ', text)

    text = re.sub(r'http\S+', ' ', text)

    text = re.sub(r'[^a-zA-Z]', ' ', text)

    text = re.sub(r'\s+', ' ', text)

    tokens = text.split()

    tokens = [word for word in tokens if word not in stop_words]

    text = " ".join(tokens)

    return text


train["clean_text"] = train["text"].apply(clean_text)
test["clean_text"] = test["text"].apply(clean_text)


OSError: [E050] Can't find model 'en_core_web_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

In [ ]:

############################################################
# 4. LABEL ENCODING
############################################################

label_encoder = LabelEncoder()

train["label_encoded"] = label_encoder.fit_transform(train["label"])

num_labels = len(label_encoder.classes_)

print("Number of Classes:", num_labels)


In [ ]:

############################################################
# 5. TRAIN TEST SPLIT
############################################################

X_train, X_val, y_train, y_val = train_test_split(
    train["clean_text"],
    train["label_encoded"],
    test_size=0.1,
    random_state=42
)


In [ ]:

############################################################
# 6. TOKENIZATION USING BERT
############################################################

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


def tokenize_data(texts):

    return tokenizer(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )


train_encodings = tokenize_data(X_train)
val_encodings = tokenize_data(X_val)


In [ ]:

############################################################
# 7. CREATE DATASET CLASS
############################################################

class ResumeDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):

        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels.iloc[idx])

        return item

    def __len__(self):

        return len(self.labels)


train_dataset = ResumeDataset(train_encodings, y_train)
val_dataset = ResumeDataset(val_encodings, y_val)


In [ ]:

############################################################
# 8. LOAD BERT MODEL
############################################################

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels
)


In [ ]:

############################################################
# 9. TRAINING ARGUMENTS
############################################################

training_args = TrainingArguments(

    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    load_best_model_at_end=True

)


In [ ]:

############################################################
# 10. TRAINER
############################################################

trainer = Trainer(

    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset

)


In [ ]:
############################################################
# 11. TRAIN MODEL
############################################################

trainer.train()


In [ ]:

############################################################
# 12. EVALUATION
############################################################

predictions = trainer.predict(val_dataset)

preds = np.argmax(predictions.predictions, axis=1)

print("Accuracy:", accuracy_score(y_val, preds))

print(classification_report(y_val, preds))


In [ ]:

############################################################
# 13. SAVE MODEL
############################################################

model.save_pretrained("resume_bert_model")
tokenizer.save_pretrained("resume_bert_model")


In [ ]:

############################################################
# 14. RAG SYSTEM (Resume Matching)
############################################################

print("Building RAG System...")

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

resume_embeddings = embedding_model.encode(
    train["clean_text"].tolist(),
    show_progress_bar=True
)


In [ ]:

############################################################
# 15. FAISS INDEX
############################################################

dimension = resume_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(resume_embeddings))

print("FAISS Index Built")


In [ ]:

############################################################
# 16. JOB DESCRIPTION MATCHING FUNCTION
############################################################

def match_resumes(job_description, top_k=5):

    job_embedding = embedding_model.encode([job_description])

    distances, indices = index.search(job_embedding, top_k)

    results = train.iloc[indices[0]]

    return results[["text", "label"]]



In [ ]:

############################################################
# 17. TEST RAG SYSTEM
############################################################

job_description = """
Looking for Data Scientist with python,
machine learning, deep learning,
NLP and pandas experience
"""

results = match_resumes(job_description)

print("Top Matching Resumes:")
print(results)



In [ ]:
############################################################
# 18. COMPLETE ATS PIPELINE
############################################################

def smart_resume_screening(job_description):

    print("Top Matching Candidates")

    results = match_resumes(job_description, top_k=10)

    return results



In [ ]:

############################################################
# 19. FINAL TEST
############################################################

job = """
Looking for web developer with
HTML CSS Javascript React
"""

final = smart_resume_screening(job)

print(final)